In [12]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"

import util.supabase_utils as su
from dotenv import load_dotenv
from tqdm.auto import tqdm

import src.config as config
import src.data_preprocessing.text_preprocessing as text_pre
from src.data_preprocessing.TextEmbedder import TextEmbedder
from src.agents.util.Retriever import Retriever
from src.agents.util.ResponseModel import ResponseModel

In [2]:

plant_expert_vqa_TRAIN= text_pre.load_csv(config.Data.PlantExpertVQA.TRAIN_FILE)
plant_expert_vqa_TEST= text_pre.load_csv(config.Data.PlantExpertVQA.TEST_FILE)
plant_expert_vqa_VAL= text_pre.load_csv(config.Data.PlantExpertVQA.VALIDATION_FILE)

# training

text_pre.preprocess_dataframe(
    plant_expert_vqa_TRAIN,
    config.Data.PlantExpertVQA.TEXT_COLUMNS,
    config.Data.PlantExpertVQA.COLUMNS_TO_REMOVE,
    config.Data.PlantExpertVQA.NA_FILL,
    ["image_path"],
    config.Data.PlantExpertVQA.ROOT,
)

# testing
text_pre.preprocess_dataframe(
    plant_expert_vqa_TEST,
    config.Data.PlantExpertVQA.TEXT_COLUMNS,
    config.Data.PlantExpertVQA.COLUMNS_TO_REMOVE,
    config.Data.PlantExpertVQA.NA_FILL,
    ["image_path"],
    config.Data.PlantExpertVQA.ROOT,
)

#validation
text_pre.preprocess_dataframe(
    plant_expert_vqa_VAL,
    config.Data.PlantExpertVQA.TEXT_COLUMNS,
    config.Data.PlantExpertVQA.COLUMNS_TO_REMOVE,
    config.Data.PlantExpertVQA.NA_FILL,
    ["image_path"],
    config.Data.PlantExpertVQA.ROOT,
)

19:39:29 [DEBUG   ]: [load_csv] Successfully loaded /mnt/c/Users/codyr/Code/AI-572/PlantQA/data/PlantExpertVQA/data/train.csv into dataframe


/mnt/c/Users/codyr/Code/AI-572/PlantQA/src/data_preprocessing/text_preprocessing.py:48: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  data= pd.read_csv(csv_path)


19:39:30 [DEBUG   ]: [load_csv] Successfully loaded /mnt/c/Users/codyr/Code/AI-572/PlantQA/data/PlantExpertVQA/data/test.csv into dataframe
19:39:30 [DEBUG   ]: [load_csv] Successfully loaded /mnt/c/Users/codyr/Code/AI-572/PlantQA/data/PlantExpertVQA/data/val.csv into dataframe


In [13]:
embedder = TextEmbedder("sentence-transformers/all-MiniLM-L6-v2")
client = su.connect_supabase()
retriever = Retriever(client, embedder)

retriever.retrieve("how do I treat leaf spot")[:1]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[{'id': 2772,
  'title': 'AMCA4',
  'body': 'be infected by foliar diseases like leaf spot, rust, or powdery mildew. To help prevent leaf spot, fallen leaves should be raked away and overhead irrigation should be avoided. Pests may include Japanese beetles, spider mites, aphids, and leaf miners (Dickert, 2010). The',
  'plant_code': 'AMCA4',
  'common_name': 'unknown',
  'scientific_name': 'Amelanchier canadensis (L.) Medik.',
  'similarity': 0.63679051399231}]

In [14]:
train_sample = (plant_expert_vqa_TRAIN.drop_duplicates(subset="question_text").sample(200, random_state=0).reset_index(drop=True))



In [15]:
model = ResponseModel(lora=True)

question = "What causes gourd anthracnose?"
print(model.generate_answer(question, retriever.retrieve(question)))

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

The information provided does not contain any specific details about what causes gourd anthracnose. Anthracnose is a fungal disease that affects various plants, including some cucurbits like gourds. It is typically caused by fungi from the genus *Glomerella* or *Colletotrichum*, such as *C. coccinea*. These fungi can infect wounds, natural openings, and healthy tissues on the plant. The disease spreads through spores produced on infected plant parts and survives in the soil between growing seasons. To manage anthracnose, it's important to practice good cultural practices, rotate crops, and apply fungicides as needed.


In [16]:
model.train_model(train_sample, retriever=retriever, epochs=1,
                  out="./models/response_models/smoke_test")

Step,Training Loss


In [17]:
import evaluation.text_evaluator as te
import pandas as pd
from src.util.logger import Logger
records = []
rows = plant_expert_vqa_VAL.sample(10, random_state=0)
for row in rows.itertuples():
    chunks = retriever.retrieve(row.question_text)
    records.append({
        "question": row.question_text,
        "image_path": row.image_path,
        "chunks": chunks,
        "num_chunks": len(chunks),
        "answer": model.generate_answer(row.question_text, chunks),
    })
answers = []
for record in records:
    answers.append(te.evaluate_response(
        record["answer"],
        record["image_path"],
        record["question"],
        chunks=record["chunks"],
    ))

answers_df = pd.DataFrame(answers)

for answer in answers:
    Logger.info(answer)

BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}, 'request_id': 'req_011CfUbBtQMuV3L4tg9r3DbE'}